# Customer Churn Analysis
Telco Customer Churn dataset (IBM Sample Data Sets, public, 7,043 customers).

## 1. Load & Clean Data

In [1]:
import pandas as pd
import sqlite3
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import json

df = pd.read_csv("Telco-Customer-Churn.csv")

# TotalCharges has some blank strings for brand-new customers; coerce to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
n_missing = df["TotalCharges"].isna().sum()
df = df.dropna(subset=["TotalCharges"])  # these are tenure=0 customers, no charges yet

df["ChurnFlag"] = (df["Churn"] == "Yes").astype(int)



## 2. SQL Analysis (via SQLite)
Queries are also available standalone in `churn_analysis.sql`.

In [2]:
import sqlite3
conn = sqlite3.connect(":memory:")
df.to_sql("customers", conn, index=False, if_exists="replace")

overall_churn_rate = pd.read_sql('''
    SELECT COUNT(*) AS total_customers,
           SUM(ChurnFlag) AS churned_customers,
           ROUND(100.0 * SUM(ChurnFlag) / COUNT(*), 2) AS churn_rate_pct
    FROM customers
''', conn)
overall_churn_rate

,total_customers,churned_customers,churn_rate_pct
0,7032,1869,26.58


In [3]:
churn_by_contract = pd.read_sql('''
    SELECT Contract, COUNT(*) AS customers, SUM(ChurnFlag) AS churned,
           ROUND(100.0 * SUM(ChurnFlag) / COUNT(*), 2) AS churn_rate_pct
    FROM customers GROUP BY Contract ORDER BY churn_rate_pct DESC
''', conn)
churn_by_contract

,Contract,customers,churned,churn_rate_pct
0,Month-to-month,3875,1655,42.71
1,One year,1472,166,11.28
2,Two year,1685,48,2.85


In [4]:
churn_by_internet_service = pd.read_sql('''
    SELECT InternetService, COUNT(*) AS customers,
           ROUND(100.0 * SUM(ChurnFlag) / COUNT(*), 2) AS churn_rate_pct,
           ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charges
    FROM customers GROUP BY InternetService ORDER BY churn_rate_pct DESC
''', conn)
churn_by_internet_service

,InternetService,customers,churn_rate_pct,avg_monthly_charges
0,Fiber optic,3096,41.89,91.50
1,DSL,2416,19.00,58.09
2,No,1520,7.43,21.08


In [5]:
churn_by_tenure_bucket = pd.read_sql('''
    SELECT CASE WHEN tenure <= 12 THEN '0-12 months'
                WHEN tenure <= 24 THEN '13-24 months'
                WHEN tenure <= 48 THEN '25-48 months'
                ELSE '49+ months' END AS tenure_bucket,
           COUNT(*) AS customers,
           ROUND(100.0 * SUM(ChurnFlag) / COUNT(*), 2) AS churn_rate_pct
    FROM customers GROUP BY tenure_bucket ORDER BY MIN(tenure)
''', conn)
churn_by_tenure_bucket

,tenure_bucket,customers,churn_rate_pct
0,0-12 months,2175,47.68
1,13-24 months,1024,28.71
2,25-48 months,1594,20.39
3,49+ months,2239,9.51


In [6]:
high_risk_segment = pd.read_sql('''
    SELECT Contract, InternetService, PaymentMethod, COUNT(*) AS customers,
           ROUND(100.0 * SUM(ChurnFlag) / COUNT(*), 2) AS churn_rate_pct
    FROM customers
    GROUP BY Contract, InternetService, PaymentMethod
    HAVING COUNT(*) >= 50
    ORDER BY churn_rate_pct DESC LIMIT 5
''', conn)
high_risk_segment

,Contract,InternetService,PaymentMethod,customers,churn_rate_pct
0,Month-to-month,Fiber optic,Electronic check,1307,60.37
1,Month-to-month,Fiber optic,Mailed check,201,50.75
2,Month-to-month,Fiber optic,Bank transfer (automatic),327,45.57
3,Month-to-month,Fiber optic,Credit card (automatic),293,41.64
4,Month-to-month,DSL,Electronic check,474,40.51


## 3. Visualizations

In [7]:
plt.figure(figsize=(6,4))
plt.bar(churn_by_contract["Contract"], churn_by_contract["churn_rate_pct"], color="#2E5C8A")
plt.title("Churn Rate by Contract Type"); plt.ylabel("Churn Rate (%)"); plt.tight_layout(); plt.show()

In [8]:
plt.figure(figsize=(6,4))
plt.bar(churn_by_tenure_bucket["tenure_bucket"], churn_by_tenure_bucket["churn_rate_pct"], color="#C0392B")
plt.title("Churn Rate by Tenure"); plt.ylabel("Churn Rate (%)"); plt.tight_layout(); plt.show()

In [9]:
plt.figure(figsize=(6,4))
plt.bar(churn_by_internet_service["InternetService"], churn_by_internet_service["churn_rate_pct"], color="#27AE60")
plt.title("Churn Rate by Internet Service Type"); plt.ylabel("Churn Rate (%)"); plt.tight_layout(); plt.show()

## 4. Findings

- Overall churn rate: **26.6%**
- Month-to-month contracts churn at **42.7%** vs **2.9%** for two-year contracts
- Fiber optic customers churn more than double DSL customers (41.9% vs 19.0%)
- 47.7% of customers with under 12 months' tenure churn
- Highest-risk segment: month-to-month + fiber optic + electronic check → **60.4% churn**

**Recommendation:** target retention offers at new fiber-optic, month-to-month customers within their first year.